# The Thermodynamic and Economic Consequences of Terafab on the United States Energy and Water through 2030

This Google Colab notebook uses **`terafab-decision-twin`** as the modeling kernel and provides a forecast-first publication-analysis layer for high-resolution scenario trajectories, temporal stress windows, declared seasonal stress variants, temporal uncertainty envelopes, validation-readiness screening, and manuscript-quality figure/table export.

**Rights and evidence boundary.** This repository is source-available, not open source. The analyses below use public examples and declared assumptions only. Outputs are conditional analytical estimates from `terafab-decision-twin`; they are not verified Terafab operating facts, engineering certifications, investment advice, permitting determinations, procurement guidance, public-policy endorsements, or evidence of Terafab affiliation, authorization, financing, construction, operation, or adoption.

**Forecast boundary.** Quarterly and monthly periods are declared model periods. Higher time resolution does not imply empirical seasonal calibration. Declared seasonal stress overlays are stress cases, not verified seasonal operating data.

## 1. Colab setup, package installation, and imports

This setup locates the repository, installs the local package in editable mode, imports `terafab-decision-twin`, and installs notebook-only plotting/data packages when they are absent. In normal Google Colab runtimes, `pandas`, `numpy`, and `matplotlib` are typically already available.

In [ ]:
from __future__ import annotations

import copy
import importlib.util
import json
import math
import os
import platform
import random
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path


def ensure_packages(packages: dict[str, str]) -> None:
    missing = [package for package, module in packages.items() if importlib.util.find_spec(module) is None]
    if missing:
        subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

ensure_packages({"pandas": "pandas", "numpy": "numpy", "matplotlib": "matplotlib"})

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents, Path('/content/terafab-decision-twin'), Path('/workspace/terafab-decision-twin')]:
    if (candidate / 'terafab_decision_twin').exists() and (candidate / 'pyproject.toml').exists():
        ROOT = candidate.resolve()
        break
else:
    raise RuntimeError('Could not locate terafab-decision-twin source. Clone or upload the repository before running this notebook.')

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", str(ROOT)])

from terafab_decision_twin.engine import MODEL_VERSION, run_scenario
from terafab_decision_twin.schema import load_scenario, validate_scenario
from strategic_simulation.monte_carlo import sample_distribution
from validation_lab import validate_result_structure, validate_reference_ranges, validation_scorecard

env_df = pd.DataFrame([{
    'run_utc': datetime.now(timezone.utc).isoformat(),
    'python': sys.version.split()[0],
    'platform': platform.platform(),
    'repository_root': str(ROOT),
    'terafab_model_version': MODEL_VERSION,
}])
display(env_df)

## 2. Time-resolution design

`terafab-decision-twin` supports `annual`, `quarterly`, and `monthly` model periods. This notebook treats time resolution as a first-class research design choice:

1. Annual scenarios provide baseline and one-year stress context.
2. Existing quarterly 2026–2030 scenarios are the primary ECM-grade forecast panel.
3. Package-native monthly clones provide higher-resolution model trajectories.
4. Declared seasonal overlays provide transparent stress cases, not empirical seasonal calibration.

In [ ]:
time_resolution_design_df = pd.DataFrame([
    {'resolution': 'annual', 'notebook_role': 'baseline and one-year stress context', 'claim_boundary': 'declared model period'},
    {'resolution': 'quarterly', 'notebook_role': 'primary 2026-2030 package forecast panel', 'claim_boundary': 'declared scenario trajectory'},
    {'resolution': 'monthly', 'notebook_role': 'high-resolution native package clone and seasonal stress overlay base', 'claim_boundary': 'declared model period; not empirical seasonal calibration'},
])
display(time_resolution_design_df)

## 3. Research questions and figure map

The notebook is forecast-first: each figure is mapped to a research question, source dataframe, and claim boundary.

In [ ]:
research_map_df = pd.DataFrame([
    {'research_question': 'How do thermodynamic consequences evolve at quarterly resolution through 2030?', 'source_dataframe': 'quarterly_thermodynamic_df', 'figure_ids': 'Fig. 2', 'claim_boundary': 'conditional forecast from package scenarios'},
    {'research_question': 'When do power, cooling, water, and wastewater margins become binding?', 'source_dataframe': 'quarterly_margin_df; constraint_calendar_df', 'figure_ids': 'Fig. 3; Fig. 4', 'claim_boundary': 'modeled feasibility screen'},
    {'research_question': 'How do water flows and permit margins evolve across forecast periods?', 'source_dataframe': 'quarterly_water_df; monthly_native_forecast_df', 'figure_ids': 'Fig. 5; Fig. 7', 'claim_boundary': 'declared water forecast'},
    {'research_question': 'How do thermodynamic losses couple to modeled cost over time?', 'source_dataframe': 'quarterly_economics_df; forecast_wide_df', 'figure_ids': 'Fig. 6', 'claim_boundary': 'scenario-conditioned thermoeconomic trajectory'},
    {'research_question': 'How do declared seasonal stress overlays change forecast stress windows?', 'source_dataframe': 'declared_seasonal_stress_forecast_df', 'figure_ids': 'Fig. 8', 'claim_boundary': 'declared stress overlay'},
    {'research_question': 'How does uncertainty propagate over time rather than only into scalar summaries?', 'source_dataframe': 'mc_period_quantiles_df; mc_margin_failure_probability_by_period_df', 'figure_ids': 'Fig. 9; Fig. 10', 'claim_boundary': 'declared-input uncertainty'},
    {'research_question': 'Which periods are worst, and why?', 'source_dataframe': 'worst_period_df', 'figure_ids': 'Fig. 11', 'claim_boundary': 'ranked model periods'},
    {'research_question': 'Which evidence and validation gaps bound interpretation?', 'source_dataframe': 'validation_readiness_df; evidence_status_df', 'figure_ids': 'Fig. 12', 'claim_boundary': 'validation-readiness only'},
])
display(research_map_df)

## 4. Publication plotting and export standard

Figures export as SVG, PDF, and high-resolution PNG. Every caption records method and evidence boundaries in a manifest.

In [ ]:
FIG_DIR = ROOT / 'publication_figures'
FIG_DIR.mkdir(exist_ok=True)

SCENARIO_COLORS = {
    'Baseline': '#0072B2',
    'Best case': '#009E73',
    'Worst case': '#D55E00',
    'Multi-year planning case': '#CC79A7',
    'Terawatt stress case': '#000000',
}
SEASON_COLORS = {'native_monthly': '#0072B2', 'summer_cooling_stress': '#D55E00', 'summer_water_stress': '#CC79A7', 'winter_grid_price_stress': '#56B4E9'}
LINE_STYLES = ['-', '--', '-.', ':', (0, (3, 1, 1, 1))]
figure_manifest: list[dict[str, str]] = []

def set_publication_style() -> None:
    plt.rcParams.update({
        'figure.dpi': 130,
        'savefig.dpi': 600,
        'font.size': 9,
        'axes.labelsize': 9,
        'axes.titlesize': 10,
        'xtick.labelsize': 8,
        'ytick.labelsize': 8,
        'legend.fontsize': 8,
        'lines.linewidth': 1.8,
        'axes.grid': True,
        'grid.alpha': 0.25,
        'grid.linewidth': 0.5,
        'axes.spines.top': False,
        'axes.spines.right': False,
        'pdf.fonttype': 42,
        'ps.fonttype': 42,
    })


def save_publication_figure(fig, stem: str, title: str, primary_claim: str, primary_metrics: str, source_dataframe: str, scenarios_used: str, method_boundary: str, evidence_boundary: str) -> dict[str, str]:
    paths = {}
    for ext in ['svg', 'pdf', 'png']:
        path = FIG_DIR / f'{stem}.{ext}'
        fig.savefig(path, bbox_inches='tight', facecolor='white')
        paths[ext] = str(path.relative_to(ROOT))
    figure_manifest.append({
        'figure_id': stem,
        'figure_title': title,
        'filename_svg': paths['svg'],
        'filename_pdf': paths['pdf'],
        'filename_png': paths['png'],
        'primary_claim': primary_claim,
        'primary_metrics': primary_metrics,
        'source_dataframe': source_dataframe,
        'scenarios_used': scenarios_used,
        'method_boundary': method_boundary,
        'evidence_boundary': evidence_boundary,
        'suggested_caption': f'{title}. {primary_claim} {method_boundary} {evidence_boundary}',
    })
    return paths


def add_panel_label(ax, label: str) -> None:
    ax.text(-0.10, 1.06, label, transform=ax.transAxes, fontsize=10, fontweight='bold', va='top')


def safe_divide(a, b):
    if a is None or b is None or pd.isna(a) or pd.isna(b) or float(b) == 0.0:
        return np.nan
    return float(a) / float(b)


def period_sort_key(label: str) -> tuple[int, int]:
    text = str(label)
    if 'Q' in text:
        year, q = text.split('Q')
        return int(year), int(q)
    if 'M' in text:
        year, m = text.split('M')
        return int(year), int(m)
    return int(text), 1

set_publication_style()

## 5. Scenario registry and validation

The quarterly 2026–2030 package scenarios are the primary forecast set. Annual baseline and terawatt stress scenarios provide context.

In [ ]:
scenario_specs = [
    ('Baseline', ROOT / 'scenarios/baseline_2026.json', 'context_annual'),
    ('Best case', ROOT / 'scenarios/best_case_2026_2030.json', 'primary_quarterly_forecast'),
    ('Worst case', ROOT / 'scenarios/worst_case_2026_2030.json', 'primary_quarterly_forecast'),
    ('Multi-year planning case', ROOT / 'scenarios/multi_year_2026_2030.json', 'primary_quarterly_forecast'),
    ('Terawatt stress case', ROOT / 'scenarios/terawatt_stress_2026.json', 'context_stress_test'),
]
scenarios = {}
registry_rows = []
for label, path, role in scenario_specs:
    scenario = load_scenario(path)
    errors = validate_scenario(scenario)
    scenarios[label] = scenario
    registry_rows.append({
        'scenario_label': label,
        'scenario_id': scenario['metadata'].get('scenario_id'),
        'scenario_title': scenario['metadata'].get('title'),
        'scenario_role': role,
        'time_step': scenario['time'].get('time_step'),
        'start_year': scenario['time'].get('start_year'),
        'end_year': scenario['time'].get('end_year'),
        'file_path': str(path.relative_to(ROOT)),
        'validation_passed': not errors,
        'validation_errors': '; '.join(errors),
        'claim_boundary': 'declared scenario input; not verified Terafab operating data',
    })
scenario_registry_df = pd.DataFrame(registry_rows)
display(scenario_registry_df)
assert scenario_registry_df['validation_passed'].all(), 'One or more scenarios failed schema validation.'

## 6. Forecast execution and high-resolution dataframes

This section runs `terafab-decision-twin` and reshapes the time-series output into forecast-first long and wide tables.

In [ ]:
def run_labeled_scenarios(scenario_map: dict[str, dict]) -> dict[str, dict]:
    return {label: run_scenario(scenario) for label, scenario in scenario_map.items()}


def result_frames(results_by_label: dict[str, dict], scenario_registry: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    registry_by_label = scenario_registry.set_index('scenario_label').to_dict('index')
    summary_rows, ts_rows, gate_rows, evidence_rows, hash_rows = [], [], [], [], []
    for label, result in results_by_label.items():
        reg = registry_by_label.get(label, {})
        scenario_id = result['metadata'].get('scenario_id')
        summary = dict(result['summary'])
        summary.update({'scenario_label': label, 'scenario_id': scenario_id, 'time_step': reg.get('time_step'), 'scenario_role': reg.get('scenario_role'), 'passed': result.get('passed'), 'model_version': MODEL_VERSION})
        summary_rows.append(summary)
        for row in result.get('time_series', []):
            out = dict(row)
            out.update({'scenario_label': label, 'scenario_id': scenario_id, 'time_step': reg.get('time_step'), 'scenario_role': reg.get('scenario_role'), 'model_version': MODEL_VERSION})
            out['period_sort'] = period_sort_key(out['label'])[0] * 100 + period_sort_key(out['label'])[1]
            ts_rows.append(out)
        for gate in result.get('gates', []):
            out = dict(gate)
            out.update({'scenario_label': label, 'scenario_id': scenario_id, 'time_step': reg.get('time_step'), 'scenario_role': reg.get('scenario_role')})
            gate_rows.append(out)
        for status, count in result.get('evidence', {}).get('status_counts', {}).items():
            evidence_rows.append({'scenario_label': label, 'scenario_id': scenario_id, 'evidence_status': status, 'count': count})
        hashes = dict(result.get('hashes', {})); hashes.update({'scenario_label': label, 'scenario_id': scenario_id}); hash_rows.append(hashes)
    return pd.DataFrame(summary_rows), pd.DataFrame(ts_rows), pd.DataFrame(gate_rows), pd.DataFrame(evidence_rows), pd.DataFrame(hash_rows)

results_by_label = run_labeled_scenarios(scenarios)
summary_df, timeseries_df, gates_df, evidence_status_df, hashes_df = result_frames(results_by_label, scenario_registry_df)

metric_registry = {
    'energy_MWh': ('MWh','power'), 'firm_capacity_margin_MW': ('MW','power'),
    'heat_rejection_required_MW': ('MW','cooling'), 'heat_rejection_margin_MW': ('MW','cooling'), 'cooling_auxiliary_energy_MWh': ('MWh','cooling'),
    'entropy_generation_MW_per_K': ('MW/K','thermodynamics'), 'exergy_destroyed_MW': ('MW','thermodynamics'), 'exergy_efficiency': ('fraction','thermodynamics'),
    'water_withdrawal_m3': ('m3','water'), 'water_consumptive_use_m3': ('m3','water'), 'wastewater_m3': ('m3','water'),
    'water_withdrawal_margin_m3_per_day': ('m3/day','water'), 'wastewater_discharge_margin_m3_per_day': ('m3/day','water'),
    'effective_yield': ('fraction','manufacturing'), 'readiness_index': ('fraction','manufacturing'), 'good_die': ('die','manufacturing'),
    'electricity_cost_USD': ('USD','economics'), 'water_cost_USD': ('USD','economics'), 'emissions_cost_USD': ('USD','economics'), 'opex_USD': ('USD','economics'),
    'public_benefit_index': ('index','policy'), 'public_burden_index': ('index','policy'), 'legitimacy_margin': ('index','policy'), 'governance_risk_index': ('index','governance'),
}
forecast_records = []
for _, row in timeseries_df.iterrows():
    for metric, (unit, subsystem) in metric_registry.items():
        if metric in row and pd.notna(row[metric]):
            forecast_records.append({
                'scenario_label': row['scenario_label'], 'scenario_id': row['scenario_id'], 'scenario_role': row['scenario_role'], 'time_step': row['time_step'],
                'year': row['year'], 'period': row['period'], 'label': row['label'], 'period_sort': row['period_sort'], 'metric': metric, 'value': row[metric], 'unit': unit,
                'subsystem': subsystem, 'model_version': MODEL_VERSION, 'evidence_boundary': 'derived from declared scenario inputs; not verified operating data',
            })
forecast_long_df = pd.DataFrame(forecast_records)
forecast_wide_df = timeseries_df.copy()

quarterly_forecast_df = forecast_wide_df[forecast_wide_df['time_step'] == 'quarterly'].copy()
quarterly_margin_df = quarterly_forecast_df[['scenario_label','scenario_id','label','year','period','period_sort','firm_capacity_margin_MW','heat_rejection_margin_MW','water_withdrawal_margin_m3_per_day','wastewater_discharge_margin_m3_per_day']].copy()
quarterly_thermodynamic_df = quarterly_forecast_df[['scenario_label','scenario_id','label','year','period','period_sort','energy_MWh','heat_rejection_required_MW','entropy_generation_MW_per_K','exergy_destroyed_MW','exergy_efficiency']].copy()
quarterly_water_df = quarterly_forecast_df[['scenario_label','scenario_id','label','year','period','period_sort','water_withdrawal_m3','water_consumptive_use_m3','wastewater_m3','water_withdrawal_margin_m3_per_day','wastewater_discharge_margin_m3_per_day']].copy()
quarterly_economics_df = quarterly_forecast_df[['scenario_label','scenario_id','label','year','period','period_sort','electricity_cost_USD','water_cost_USD','emissions_cost_USD','opex_USD','exergy_destroyed_MW']].copy()

display(summary_df[['scenario_label','time_step','time_steps','energy_MWh','water_withdrawal_m3','total_cost_USD','passed']])
display(forecast_long_df.head())

## 7. Derived forecast indicators and stress-window analytics

These tables identify binding periods and normalize raw outputs for cross-scenario comparison.

In [ ]:
derived_forecast_df = forecast_wide_df.copy()
derived_forecast_df['heat_rejection_to_load_ratio'] = derived_forecast_df.apply(lambda r: safe_divide(r.get('heat_rejection_required_MW'), r.get('site_electric_load_MW')), axis=1)
derived_forecast_df['water_withdrawal_per_MWh'] = derived_forecast_df.apply(lambda r: safe_divide(r.get('water_withdrawal_m3'), r.get('energy_MWh')), axis=1)
derived_forecast_df['wastewater_per_MWh'] = derived_forecast_df.apply(lambda r: safe_divide(r.get('wastewater_m3'), r.get('energy_MWh')), axis=1)
derived_forecast_df['opex_per_MWh'] = derived_forecast_df.apply(lambda r: safe_divide(r.get('opex_USD'), r.get('energy_MWh')), axis=1)
derived_forecast_df['exergy_destroyed_per_MWh'] = derived_forecast_df.apply(lambda r: safe_divide(r.get('exergy_destroyed_MW'), r.get('energy_MWh')), axis=1)
derived_forecast_df['negative_power_margin'] = derived_forecast_df['firm_capacity_margin_MW'] < 0
derived_forecast_df['negative_heat_margin'] = derived_forecast_df['heat_rejection_margin_MW'] < 0
derived_forecast_df['negative_water_margin'] = derived_forecast_df['water_withdrawal_margin_m3_per_day'] < 0
derived_forecast_df['negative_wastewater_margin'] = derived_forecast_df['wastewater_discharge_margin_m3_per_day'] < 0
derived_forecast_df['failed_margin_count'] = derived_forecast_df[['negative_power_margin','negative_heat_margin','negative_water_margin','negative_wastewater_margin']].sum(axis=1)

def dominant_constraint(row):
    margins = {
        'power': row['firm_capacity_margin_MW'],
        'cooling': row['heat_rejection_margin_MW'],
        'water': row['water_withdrawal_margin_m3_per_day'],
        'wastewater': row['wastewater_discharge_margin_m3_per_day'],
    }
    return min(margins, key=lambda key: margins[key])

derived_forecast_df['dominant_constraint'] = derived_forecast_df.apply(dominant_constraint, axis=1)
stress_window_df = derived_forecast_df[['scenario_label','scenario_id','time_step','label','year','period','period_sort','firm_capacity_margin_MW','heat_rejection_margin_MW','water_withdrawal_margin_m3_per_day','wastewater_discharge_margin_m3_per_day','exergy_destroyed_MW','opex_USD','legitimacy_margin','failed_margin_count','dominant_constraint']].copy()

worst_rows = []
for label, grp in stress_window_df.groupby('scenario_label'):
    for metric, direction in [
        ('firm_capacity_margin_MW','min'), ('heat_rejection_margin_MW','min'), ('water_withdrawal_margin_m3_per_day','min'), ('wastewater_discharge_margin_m3_per_day','min'),
        ('exergy_destroyed_MW','max'), ('opex_USD','max'), ('failed_margin_count','max')
    ]:
        idx = grp[metric].idxmin() if direction == 'min' else grp[metric].idxmax()
        row = grp.loc[idx]
        worst_rows.append({'scenario_label': label, 'stress_metric': metric, 'worst_label': row['label'], 'worst_value': row[metric], 'dominant_constraint': row['dominant_constraint']})
worst_period_df = pd.DataFrame(worst_rows)
constraint_calendar_df = stress_window_df[stress_window_df['time_step'] == 'quarterly'].copy()

def longest_failure_run(grp):
    best = current = 0
    for failed in grp.sort_values('period_sort')['failed_margin_count'].gt(0):
        current = current + 1 if failed else 0
        best = max(best, current)
    return best
consecutive_failure_df = stress_window_df.groupby('scenario_label').apply(longest_failure_run, include_groups=False).reset_index(name='longest_consecutive_failed_periods')

display(stress_window_df.head())
display(worst_period_df)
display(consecutive_failure_df)

## 8. Monthly native forecasts and declared seasonal stress overlays

Package-native monthly forecasts are produced by cloning selected package scenarios and setting `time_step = "monthly"`. Declared seasonal overlays are transparent notebook stress cases applied to package-generated monthly trajectories; they are not verified seasonal operating forecasts.

In [ ]:
def clone_with_time_step(scenario: dict, time_step: str, scenario_id_suffix: str, title_suffix: str) -> dict:
    out = copy.deepcopy(scenario)
    out['time']['time_step'] = time_step
    out['metadata']['scenario_id'] = f"{out['metadata'].get('scenario_id')}_{scenario_id_suffix}"
    out['metadata']['title'] = f"{out['metadata'].get('title')} — {title_suffix}"
    out['metadata']['scenario_purpose'] = f"Notebook-derived {time_step} forecast clone; declared model resolution, not verified operating data."
    return out

monthly_scenarios = {
    'Best case monthly native': clone_with_time_step(scenarios['Best case'], 'monthly', 'monthly_native', 'monthly native forecast'),
    'Worst case monthly native': clone_with_time_step(scenarios['Worst case'], 'monthly', 'monthly_native', 'monthly native forecast'),
    'Multi-year monthly native': clone_with_time_step(scenarios['Multi-year planning case'], 'monthly', 'monthly_native', 'monthly native forecast'),
}
monthly_registry_df = pd.DataFrame([{
    'scenario_label': label, 'scenario_id': scenario['metadata']['scenario_id'], 'scenario_title': scenario['metadata']['title'], 'scenario_role': 'monthly_native_forecast',
    'time_step': scenario['time']['time_step'], 'start_year': scenario['time']['start_year'], 'end_year': scenario['time']['end_year'], 'file_path': 'notebook-derived clone',
    'validation_passed': not validate_scenario(scenario), 'validation_errors': '; '.join(validate_scenario(scenario)), 'claim_boundary': 'package-native monthly clone; declared model period'
} for label, scenario in monthly_scenarios.items()])
display(monthly_registry_df)
assert monthly_registry_df['validation_passed'].all(), 'Monthly clone validation failed.'
monthly_results = run_labeled_scenarios(monthly_scenarios)
_, monthly_native_forecast_df, _, _, _ = result_frames(monthly_results, monthly_registry_df)

seasonal_profiles = pd.DataFrame([
    {'month': 1, 'season': 'winter', 'cooling_multiplier': 0.96, 'water_multiplier': 0.98, 'opex_multiplier': 1.05},
    {'month': 2, 'season': 'winter', 'cooling_multiplier': 0.96, 'water_multiplier': 0.98, 'opex_multiplier': 1.05},
    {'month': 3, 'season': 'spring', 'cooling_multiplier': 1.00, 'water_multiplier': 1.00, 'opex_multiplier': 1.00},
    {'month': 4, 'season': 'spring', 'cooling_multiplier': 1.00, 'water_multiplier': 1.00, 'opex_multiplier': 1.00},
    {'month': 5, 'season': 'spring', 'cooling_multiplier': 1.03, 'water_multiplier': 1.02, 'opex_multiplier': 1.00},
    {'month': 6, 'season': 'summer', 'cooling_multiplier': 1.10, 'water_multiplier': 1.08, 'opex_multiplier': 1.08},
    {'month': 7, 'season': 'summer', 'cooling_multiplier': 1.15, 'water_multiplier': 1.10, 'opex_multiplier': 1.10},
    {'month': 8, 'season': 'summer', 'cooling_multiplier': 1.14, 'water_multiplier': 1.10, 'opex_multiplier': 1.10},
    {'month': 9, 'season': 'fall', 'cooling_multiplier': 1.05, 'water_multiplier': 1.04, 'opex_multiplier': 1.02},
    {'month': 10, 'season': 'fall', 'cooling_multiplier': 1.00, 'water_multiplier': 1.00, 'opex_multiplier': 1.00},
    {'month': 11, 'season': 'fall', 'cooling_multiplier': 0.98, 'water_multiplier': 0.99, 'opex_multiplier': 1.00},
    {'month': 12, 'season': 'winter', 'cooling_multiplier': 0.96, 'water_multiplier': 0.98, 'opex_multiplier': 1.06},
])
base_monthly = monthly_native_forecast_df[monthly_native_forecast_df['scenario_label'] == 'Multi-year monthly native'].copy()
base_monthly['month'] = base_monthly['period'].astype(int)
declared_seasonal_stress_forecast_df = base_monthly.merge(seasonal_profiles, on='month', how='left')
declared_seasonal_stress_forecast_df['scenario_label'] = 'Multi-year declared seasonal stress overlay'
declared_seasonal_stress_forecast_df['seasonal_heat_rejection_required_MW'] = declared_seasonal_stress_forecast_df['heat_rejection_required_MW'] * declared_seasonal_stress_forecast_df['cooling_multiplier']
declared_seasonal_stress_forecast_df['seasonal_water_withdrawal_m3'] = declared_seasonal_stress_forecast_df['water_withdrawal_m3'] * declared_seasonal_stress_forecast_df['water_multiplier']
declared_seasonal_stress_forecast_df['seasonal_opex_USD'] = declared_seasonal_stress_forecast_df['opex_USD'] * declared_seasonal_stress_forecast_df['opex_multiplier']
declared_seasonal_stress_forecast_df['claim_boundary'] = 'declared seasonal stress overlay on package-native monthly outputs; not empirical seasonal calibration'

display(monthly_native_forecast_df.head())
display(declared_seasonal_stress_forecast_df[['label','season','seasonal_heat_rejection_required_MW','seasonal_water_withdrawal_m3','seasonal_opex_USD','claim_boundary']].head(12))

## 9. Temporal Monte Carlo forecast envelopes

This notebook-level wrapper stores every run's time series, enabling per-period uncertainty envelopes and failure probabilities rather than scalar-only Monte Carlo summaries.

In [ ]:
def get_path(root: dict, dotted_path: str):
    current = root
    for part in dotted_path.split('.'):
        current = current[part]
    return current


def set_path(root: dict, dotted_path: str, value, spec: dict) -> None:
    parts = dotted_path.split('.')
    current = root
    for part in parts[:-1]:
        current = current[part]
    leaf = parts[-1]
    existing = current[leaf]
    if isinstance(existing, dict) and 'value' in existing:
        updated = dict(existing)
        updated['value'] = value
        updated['status'] = spec.get('status', updated.get('status', 'scenario_assumption'))
        updated['unit'] = spec.get('unit', updated.get('unit', ''))
        updated['notes'] = f"Temporal Monte Carlo sampled value for {dotted_path}; not verified operating data."
        current[leaf] = updated
    else:
        current[leaf] = value

mc_parameters = {
    'energy.site_electric_load_MW': {'distribution': 'triangular', 'low': 120, 'mode': 150, 'high': 190, 'unit': 'MW', 'status': 'scenario_assumption'},
    'energy.firm_capacity_MW': {'distribution': 'triangular', 'low': 170, 'mode': 200, 'high': 240, 'unit': 'MW', 'status': 'scenario_assumption'},
    'cooling.heat_rejection_capacity_MW': {'distribution': 'triangular', 'low': 140, 'mode': 170, 'high': 220, 'unit': 'MW', 'status': 'scenario_assumption'},
    'water.withdrawal_m3_per_MWh': {'distribution': 'triangular', 'low': 0.8, 'mode': 1.2, 'high': 1.8, 'unit': 'm3/MWh', 'status': 'scenario_assumption'},
    'water.permit_withdrawal_m3_per_day': {'distribution': 'triangular', 'low': 3500, 'mode': 5000, 'high': 6500, 'unit': 'm3/day', 'status': 'scenario_assumption'},
    'water.permit_discharge_m3_per_day': {'distribution': 'triangular', 'low': 1800, 'mode': 2500, 'high': 3500, 'unit': 'm3/day', 'status': 'scenario_assumption'},
    'manufacturing.baseline_yield': {'distribution': 'triangular', 'low': 0.55, 'mode': 0.72, 'high': 0.85, 'unit': 'fraction', 'status': 'scenario_assumption'},
    'economics.electricity_price_USD_per_MWh': {'distribution': 'triangular', 'low': 45, 'mode': 75, 'high': 120, 'unit': 'USD/MWh', 'status': 'scenario_assumption'},
}
uncertainty_distribution_df = pd.DataFrame([{'dotted_path': path, **spec, 'claim_boundary': 'declared input uncertainty only'} for path, spec in mc_parameters.items()])

def run_temporal_monte_carlo(base_scenario: dict, parameters: dict, runs: int = 120, seed: int = 2030) -> tuple[pd.DataFrame, pd.DataFrame]:
    rng = random.Random(seed)
    ts_rows, sample_rows = [], []
    for run_index in range(runs):
        scenario_i = copy.deepcopy(base_scenario)
        sample_record = {'run_index': run_index}
        for path, spec in parameters.items():
            get_path(scenario_i, path)
            value = sample_distribution(spec, rng)
            set_path(scenario_i, path, value, spec)
            sample_record[path] = value
        result = run_scenario(scenario_i)
        sample_record['passed'] = bool(result.get('passed'))
        sample_rows.append(sample_record)
        for row in result['time_series']:
            out = dict(row)
            out.update({'run_index': run_index, 'scenario_label': 'Multi-year temporal Monte Carlo', 'scenario_id': result['metadata']['scenario_id'], 'period_sort': period_sort_key(row['label'])[0] * 100 + period_sort_key(row['label'])[1]})
            ts_rows.append(out)
    return pd.DataFrame(ts_rows), pd.DataFrame(sample_rows)

MC_TEMPORAL_RUNS = int(os.environ.get('TERAFAB_TEMPORAL_MC_RUNS', '120'))
mc_timeseries_df, mc_samples_df = run_temporal_monte_carlo(scenarios['Multi-year planning case'], mc_parameters, runs=MC_TEMPORAL_RUNS)
mc_metrics = ['heat_rejection_required_MW','water_withdrawal_m3','opex_USD','firm_capacity_margin_MW','heat_rejection_margin_MW','water_withdrawal_margin_m3_per_day','wastewater_discharge_margin_m3_per_day']
quantile_rows = []
for (label, year, period, period_sort), grp in mc_timeseries_df.groupby(['label','year','period','period_sort']):
    for metric in mc_metrics:
        q = grp[metric].quantile([0.10, 0.50, 0.90])
        quantile_rows.append({'label': label, 'year': year, 'period': period, 'period_sort': period_sort, 'metric': metric, 'p10': q.loc[0.10], 'p50': q.loc[0.50], 'p90': q.loc[0.90]})
mc_period_quantiles_df = pd.DataFrame(quantile_rows)
failure_rows = []
for (label, year, period, period_sort), grp in mc_timeseries_df.groupby(['label','year','period','period_sort']):
    failure_rows.append({
        'label': label, 'year': year, 'period': period, 'period_sort': period_sort,
        'power_failure_probability': grp['firm_capacity_margin_MW'].lt(0).mean(),
        'heat_failure_probability': grp['heat_rejection_margin_MW'].lt(0).mean(),
        'water_failure_probability': grp['water_withdrawal_margin_m3_per_day'].lt(0).mean(),
        'wastewater_failure_probability': grp['wastewater_discharge_margin_m3_per_day'].lt(0).mean(),
    })
mc_margin_failure_probability_by_period_df = pd.DataFrame(failure_rows)
mc_stress_window_probability_df = mc_margin_failure_probability_by_period_df.copy()
mc_stress_window_probability_df['any_margin_failure_probability'] = 1 - (1 - mc_stress_window_probability_df[['power_failure_probability','heat_failure_probability','water_failure_probability','wastewater_failure_probability']]).prod(axis=1)

display(uncertainty_distribution_df)
display(mc_period_quantiles_df.head())
display(mc_margin_failure_probability_by_period_df.head())

## 10. Forecast-grade figures

The figures below emphasize temporal forecasts, stress windows, uncertainty envelopes, and claim boundaries instead of simple scenario totals.

In [ ]:
# Fig. 1: time-resolution scenario map
fig, ax = plt.subplots(figsize=(7.2, 3.0))
plot_df = pd.concat([scenario_registry_df, monthly_registry_df], ignore_index=True)
y = np.arange(len(plot_df))
starts = plot_df['start_year'].astype(float)
ends = plot_df['end_year'].astype(float)
colors = [SCENARIO_COLORS.get(label.replace(' monthly native',''), '#666666') for label in plot_df['scenario_label']]
ax.barh(y, ends - starts + 1, left=starts, color=colors, alpha=0.8)
ax.set_yticks(y); ax.set_yticklabels(plot_df['scenario_label'])
ax.set_xlabel('Modeled year')
ax.set_title('Figure 1. Time-resolution scenario map')
for idx, row in plot_df.iterrows():
    ax.text(float(row['end_year']) + 0.05, idx, f"{row['time_step']} / {row['scenario_role']}", va='center', fontsize=7)
ax.invert_yaxis(); fig.tight_layout()
save_publication_figure(fig, 'fig_01_time_resolution_map', 'Time-resolution scenario map', 'Annual, quarterly, and monthly scenario roles are separated before forecast analysis.', 'time_step; scenario_role', 'scenario_registry_df; monthly_registry_df', ', '.join(plot_df['scenario_label']), 'Scenario JSON files and notebook clones are schema checked before execution.', 'Declared model periods, not empirical measurement cadence.')
plt.show()

# Fig. 2: quarterly thermodynamic forecast
fig, axes = plt.subplots(3, 1, figsize=(7.2, 7.2), sharex=True)
for ax, metric, ylabel, panel in zip(axes, ['heat_rejection_required_MW','entropy_generation_MW_per_K','exergy_destroyed_MW'], ['Heat rejection (MW)','Entropy generation (MW/K)','Exergy destroyed (MW)'], ['(a)','(b)','(c)']):
    for i, (label, grp) in enumerate(quarterly_thermodynamic_df.groupby('scenario_label')):
        grp = grp.sort_values('period_sort')
        ax.plot(grp['label'], grp[metric], color=SCENARIO_COLORS.get(label), linestyle=LINE_STYLES[i % len(LINE_STYLES)], label=label)
    ax.set_ylabel(ylabel); add_panel_label(ax, panel)
axes[-1].tick_params(axis='x', rotation=60)
axes[0].legend(frameon=False, ncol=2)
fig.suptitle('Figure 2. Quarterly thermodynamic forecast', y=0.995); fig.tight_layout()
save_publication_figure(fig, 'fig_02_quarterly_thermodynamic_forecast', 'Quarterly thermodynamic forecast', 'Thermodynamic burdens are plotted at package-native quarterly resolution through 2030.', 'heat_rejection_required_MW; entropy_generation_MW_per_K; exergy_destroyed_MW', 'quarterly_thermodynamic_df', ', '.join(quarterly_thermodynamic_df['scenario_label'].unique()), 'Deterministic run_scenario outputs.', 'Conditional on declared package scenarios.')
plt.show()

# Fig. 3: feasibility margin heatmap
margin_metrics = ['firm_capacity_margin_MW','heat_rejection_margin_MW','water_withdrawal_margin_m3_per_day','wastewater_discharge_margin_m3_per_day']
heat_df = quarterly_margin_df.copy()
heat_df['any_margin_failure'] = heat_df[margin_metrics].lt(0).sum(axis=1)
pivot = heat_df.pivot_table(index='scenario_label', columns='label', values='any_margin_failure', aggfunc='max').fillna(0)
pivot = pivot.reindex(columns=sorted(pivot.columns, key=lambda x: period_sort_key(x)))
fig, ax = plt.subplots(figsize=(7.2, 2.8))
im = ax.imshow(pivot.values, aspect='auto', cmap='YlOrRd', vmin=0, vmax=4)
ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns, rotation=60, ha='right')
ax.set_yticks(range(len(pivot.index))); ax.set_yticklabels(pivot.index)
ax.set_title('Figure 3. Quarterly feasibility margin heatmap')
fig.colorbar(im, ax=ax, label='Failed margin count')
fig.tight_layout()
save_publication_figure(fig, 'fig_03_quarterly_feasibility_margin_heatmap', 'Quarterly feasibility margin heatmap', 'The heatmap counts modeled power, cooling, water, and wastewater margin failures by quarter.', 'failed margin count', 'quarterly_margin_df', ', '.join(pivot.index), 'Deterministic margin screen.', 'Not permitting or engineering certification.')
plt.show()

# Fig. 4: constraint calendar
fig, ax = plt.subplots(figsize=(7.2, 3.2))
calendar = constraint_calendar_df.pivot_table(index='scenario_label', columns='label', values='dominant_constraint', aggfunc='first')
calendar = calendar.reindex(columns=sorted(calendar.columns, key=lambda x: period_sort_key(x)))
constraint_codes = {'power': 0, 'cooling': 1, 'water': 2, 'wastewater': 3}
calendar_code = calendar.replace(constraint_codes).astype(float)
im = ax.imshow(calendar_code.values, aspect='auto', cmap='viridis', vmin=0, vmax=3)
ax.set_xticks(range(len(calendar_code.columns))); ax.set_xticklabels(calendar_code.columns, rotation=60, ha='right')
ax.set_yticks(range(len(calendar_code.index))); ax.set_yticklabels(calendar_code.index)
ax.set_title('Figure 4. Dominant constraint calendar')
fig.colorbar(im, ax=ax, ticks=list(constraint_codes.values()), label='0=power, 1=cooling, 2=water, 3=wastewater')
fig.tight_layout()
save_publication_figure(fig, 'fig_04_constraint_calendar', 'Dominant constraint calendar', 'The calendar identifies the lowest-margin subsystem by scenario and quarter.', 'dominant_constraint', 'constraint_calendar_df', ', '.join(calendar_code.index), 'Derived from deterministic package margins.', 'Lowest modeled margin is a screen, not proof of binding real-world constraint.')
plt.show()

In [ ]:
# Fig. 5: water forecast fan
fig, axes = plt.subplots(3, 1, figsize=(7.2, 7.0), sharex=True)
for ax, metric, ylabel, panel in zip(axes, ['water_withdrawal_m3','water_consumptive_use_m3','wastewater_m3'], ['Withdrawal (m3)','Consumptive use (m3)','Wastewater (m3)'], ['(a)','(b)','(c)']):
    for i, (label, grp) in enumerate(quarterly_water_df.groupby('scenario_label')):
        grp = grp.sort_values('period_sort')
        ax.plot(grp['label'], grp[metric], color=SCENARIO_COLORS.get(label), linestyle=LINE_STYLES[i % len(LINE_STYLES)], label=label)
    ax.set_ylabel(ylabel); add_panel_label(ax, panel)
axes[-1].tick_params(axis='x', rotation=60)
axes[0].legend(frameon=False, ncol=2)
fig.suptitle('Figure 5. Quarterly water forecast fan', y=0.995); fig.tight_layout()
save_publication_figure(fig, 'fig_05_water_forecast_fan', 'Quarterly water forecast fan', 'Water withdrawal, consumptive use, and wastewater are forecast at quarterly resolution.', 'water_withdrawal_m3; water_consumptive_use_m3; wastewater_m3', 'quarterly_water_df', ', '.join(quarterly_water_df['scenario_label'].unique()), 'Deterministic water outputs from run_scenario().', 'Scenario-conditioned model output, not measured withdrawal.')
plt.show()

# Fig. 6: thermoeconomic trajectory
fig, ax = plt.subplots(figsize=(5.2, 3.8))
for label, grp in quarterly_economics_df.groupby('scenario_label'):
    grp = grp.sort_values('period_sort')
    ax.plot(grp['exergy_destroyed_MW'], grp['opex_USD'] / 1e6, marker='o', markersize=3, color=SCENARIO_COLORS.get(label), label=label)
ax.set_xlabel('Exergy destroyed (MW)')
ax.set_ylabel('Quarterly opex (million USD)')
ax.set_title('Figure 6. Thermoeconomic trajectory')
ax.legend(frameon=False)
fig.tight_layout()
save_publication_figure(fig, 'fig_06_thermoeconomic_trajectory', 'Thermoeconomic trajectory', 'Quarterly paths compare thermodynamic losses with modeled opex over time.', 'exergy_destroyed_MW; opex_USD', 'quarterly_economics_df', ', '.join(quarterly_economics_df['scenario_label'].unique()), 'Cross-period deterministic association.', 'Not empirical causal inference.')
plt.show()

# Fig. 7: monthly native forecast
fig, ax = plt.subplots(figsize=(7.2, 3.4))
for i, (label, grp) in enumerate(monthly_native_forecast_df.groupby('scenario_label')):
    grp = grp.sort_values('period_sort')
    ax.plot(grp['label'], grp['heat_rejection_required_MW'], color=SCENARIO_COLORS.get(label.replace(' monthly native',''), '#555555'), linestyle=LINE_STYLES[i % len(LINE_STYLES)], label=label)
ax.set_ylabel('Heat rejection (MW)')
ax.set_title('Figure 7. Package-native monthly heat forecast')
ax.set_xticks(ax.get_xticks()[::max(1, len(ax.get_xticks()) // 12)])
ax.tick_params(axis='x', rotation=60)
ax.legend(frameon=False, fontsize=6)
fig.tight_layout()
save_publication_figure(fig, 'fig_07_monthly_native_forecast', 'Package-native monthly heat forecast', 'Monthly clone scenarios expose higher-resolution model periods using the package time axis.', 'heat_rejection_required_MW', 'monthly_native_forecast_df', ', '.join(monthly_native_forecast_df['scenario_label'].unique()), 'Notebook-derived monthly clones executed with run_scenario().', 'Monthly period resolution is not empirical seasonal calibration.')
plt.show()

# Fig. 8: declared seasonal stress comparison
fig, axes = plt.subplots(3, 1, figsize=(7.2, 7.0), sharex=True)
seasonal_plot = declared_seasonal_stress_forecast_df.sort_values('period_sort')
base_plot = base_monthly.sort_values('period_sort')
for ax, base_metric, seasonal_metric, ylabel, panel in zip(
    axes,
    ['heat_rejection_required_MW','water_withdrawal_m3','opex_USD'],
    ['seasonal_heat_rejection_required_MW','seasonal_water_withdrawal_m3','seasonal_opex_USD'],
    ['Heat rejection (MW)','Water withdrawal (m3)','Opex (USD)'],
    ['(a)','(b)','(c)']
):
    ax.plot(base_plot['label'], base_plot[base_metric], label='package-native monthly', color='#0072B2')
    ax.plot(seasonal_plot['label'], seasonal_plot[seasonal_metric], label='declared seasonal overlay', color='#D55E00', linestyle='--')
    ax.set_ylabel(ylabel); add_panel_label(ax, panel)
axes[-1].tick_params(axis='x', rotation=60)
axes[0].legend(frameon=False)
fig.suptitle('Figure 8. Declared seasonal stress comparison', y=0.995); fig.tight_layout()
save_publication_figure(fig, 'fig_08_declared_seasonal_stress_comparison', 'Declared seasonal stress comparison', 'Declared seasonal overlays stress heat, water, and opex relative to package-native monthly outputs.', 'seasonal heat; seasonal water; seasonal opex', 'declared_seasonal_stress_forecast_df', 'Multi-year monthly native', 'Postprocessed declared stress overlay on package outputs.', 'Not verified seasonal operating data.')
plt.show()

In [ ]:
# Fig. 9: temporal uncertainty envelopes
fig, axes = plt.subplots(3, 1, figsize=(7.2, 7.2), sharex=True)
for ax, metric, ylabel, panel in zip(axes, ['heat_rejection_required_MW','water_withdrawal_m3','opex_USD'], ['Heat rejection (MW)','Water withdrawal (m3)','Opex (USD)'], ['(a)','(b)','(c)']):
    q = mc_period_quantiles_df[mc_period_quantiles_df['metric'] == metric].sort_values('period_sort')
    x = np.arange(len(q))
    ax.fill_between(x, q['p10'], q['p90'], color='#999999', alpha=0.25, label='p10-p90')
    ax.plot(x, q['p50'], color='#0072B2', label='p50')
    ax.set_ylabel(ylabel); add_panel_label(ax, panel)
axes[-1].set_xticks(np.arange(len(q)))
axes[-1].set_xticklabels(q['label'], rotation=60, ha='right')
axes[0].legend(frameon=False)
fig.suptitle('Figure 9. Temporal Monte Carlo uncertainty envelopes', y=0.995); fig.tight_layout()
save_publication_figure(fig, 'fig_09_forecast_uncertainty_envelope', 'Temporal Monte Carlo uncertainty envelopes', 'Per-period p10-p50-p90 envelopes show declared uncertainty propagation over the quarterly forecast horizon.', 'heat_rejection_required_MW; water_withdrawal_m3; opex_USD', 'mc_period_quantiles_df', 'Multi-year temporal Monte Carlo', 'Notebook-level temporal Monte Carlo around run_scenario().', 'Declared-input uncertainty, not empirical confidence interval.')
plt.show()

# Fig. 10: failure probability by period
fig, ax = plt.subplots(figsize=(7.2, 3.6))
fp = mc_margin_failure_probability_by_period_df.sort_values('period_sort')
for metric, label, color in [
    ('power_failure_probability','power','#0072B2'), ('heat_failure_probability','cooling','#D55E00'), ('water_failure_probability','water','#009E73'), ('wastewater_failure_probability','wastewater','#CC79A7')
]:
    ax.plot(fp['label'], fp[metric], marker='o', markersize=3, label=label, color=color)
ax.set_ylabel('Failure probability')
ax.set_title('Figure 10. Period failure probability')
ax.tick_params(axis='x', rotation=60)
ax.legend(frameon=False, ncol=2)
fig.tight_layout()
save_publication_figure(fig, 'fig_10_period_failure_probability', 'Period failure probability', 'Failure probabilities show when declared uncertainty most often breaches modeled margins.', 'power/cooling/water/wastewater failure probability', 'mc_margin_failure_probability_by_period_df', 'Multi-year temporal Monte Carlo', 'Per-period failure probability from sampled run_scenario outputs.', 'Scenario-conditioned probability, not real-world probability.')
plt.show()

# Fig. 11: worst-period decomposition
fig, ax = plt.subplots(figsize=(7.2, 3.8))
wp = worst_period_df.copy()
wp['label_metric'] = wp['scenario_label'] + ' / ' + wp['stress_metric']
ax.barh(wp['label_metric'], wp['worst_value'], color='#56B4E9')
ax.set_title('Figure 11. Worst-period decomposition')
ax.set_xlabel('Worst-period value in native units')
fig.tight_layout()
save_publication_figure(fig, 'fig_11_worst_period_decomposition', 'Worst-period decomposition', 'Worst forecast periods identify when and why each scenario reaches maximum stress or minimum margin.', 'worst_value by stress metric', 'worst_period_df', ', '.join(worst_period_df['scenario_label'].unique()), 'Derived ranking from deterministic period outputs.', 'Ranking reflects model-period outputs, not observed operating events.')
plt.show()

## 11. Validation/evidence forecast boundary

Validation-readiness is structural and public-reference screening only. It does not convert assumptions into verified facts.

In [ ]:
reference_ranges = json.loads((ROOT / 'validation_lab/reference_ranges.json').read_text())
validation_rows, unknown_rows = [], []
for label, result in results_by_label.items():
    structure = validate_result_structure(result)
    ranges = validate_reference_ranges(result['summary'], reference_ranges)
    score = validation_scorecard([structure, ranges])
    validation_rows.append({
        'scenario_label': label,
        'scenario_id': result['metadata']['scenario_id'],
        'structural_passed': structure['passed'],
        'reference_ranges_passed': ranges['passed'],
        'reference_range_flags': len(ranges['range_flags']),
        **score,
    })
    unknown_rows.append({'scenario_label': label, 'unknown_count': len(result['unknowns']['unresolved_variables']), 'underdetermined': result['unknowns']['underdetermined']})
validation_readiness_df = pd.DataFrame(validation_rows)
unknowns_df = pd.DataFrame(unknown_rows)
unsupported_claims_df = pd.DataFrame({'unsupported_claim': [
    'Verified Terafab operating data', 'Official Terafab endorsement or affiliation', 'Actual U.S. national energy or water impact as fact',
    'Permitting approval or environmental compliance', 'Investment merit or financing forecast', 'Construction, operation, adoption, or procurement proof',
    'External empirical validation beyond structural and public-reference screening', 'Verified seasonal operating forecast'
]})
highest_value_next_data_df = pd.DataFrame({'next_data_need': [
    'Site-specific power capacity and tariff structure', 'Cooling design basis and heat-rejection system data', 'Permitted water withdrawal and discharge values',
    'Time-varying or seasonal operating profiles', 'Yield, packaging, and qualification calibration data', 'External benchmark or reference dataset for validation'
]})

fig, axes = plt.subplots(1, 3, figsize=(7.2, 3.1))
epivot = evidence_status_df.pivot_table(index='scenario_label', columns='evidence_status', values='count', aggfunc='sum', fill_value=0)
epivot.plot(kind='bar', stacked=True, ax=axes[0], colormap='tab20')
axes[0].set_title('Evidence status'); axes[0].set_ylabel('Count'); axes[0].legend(frameon=False, fontsize=5)
validation_readiness_df.set_index('scenario_label')['score'].plot(kind='bar', ax=axes[1], color='#009E73')
axes[1].set_title('Validation score'); axes[1].set_ylim(0, 1.05)
unknowns_df.set_index('scenario_label')['unknown_count'].plot(kind='bar', ax=axes[2], color='#D55E00')
axes[2].set_title('Unresolved variables'); axes[2].set_ylabel('Count')
for ax in axes:
    ax.tick_params(axis='x', labelrotation=60)
fig.suptitle('Figure 12. Validation/evidence forecast boundary', y=1.03); fig.tight_layout()
save_publication_figure(fig, 'fig_12_validation_evidence_forecast_boundary', 'Validation/evidence forecast boundary', 'Evidence composition, validation-readiness scores, and unknown counts bound interpretation of the forecast set.', 'evidence status; validation score; unknown count', 'evidence_status_df; validation_readiness_df; unknowns_df', ', '.join(summary_df['scenario_label'].unique()), 'validation_lab structural and public-reference screening.', 'Not external validation, certification, permitting approval, or verified Terafab data.')
plt.show()

display(validation_readiness_df)
display(unsupported_claims_df)
display(highest_value_next_data_df)

## 12. High-resolution forecast table exports and figure manifest

The export tables are intended for manuscript supplements and auditability.

In [ ]:
exports = {
    'quarterly_forecast_all_metrics.csv': quarterly_forecast_df,
    'quarterly_margin_calendar.csv': constraint_calendar_df,
    'quarterly_stress_window_ranking.csv': worst_period_df,
    'monthly_native_forecast_all_metrics.csv': monthly_native_forecast_df,
    'declared_seasonal_stress_forecast.csv': declared_seasonal_stress_forecast_df,
    'temporal_monte_carlo_quantiles.csv': mc_period_quantiles_df,
    'period_failure_probability.csv': mc_margin_failure_probability_by_period_df,
    'scenario_registry.csv': scenario_registry_df,
    'validation_readiness.csv': validation_readiness_df,
}
for filename, frame in exports.items():
    frame.to_csv(FIG_DIR / filename, index=False)
figure_manifest_df = pd.DataFrame(figure_manifest)
figure_manifest_df.to_csv(FIG_DIR / 'figure_manifest.csv', index=False)
display(figure_manifest_df)
print(f'Exported {len(exports)} tables and {len(figure_manifest_df)} figures to {FIG_DIR.relative_to(ROOT)}')

## 13. Time resolution is not seasonal calibration

`terafab-decision-twin` supports annual, quarterly, and monthly model periods. Higher time resolution improves forecast granularity but does not by itself create verified seasonal calibration. The deterministic scenario model uses declared inputs across generated model periods. Monthly clones and seasonal stress overlays in this notebook are declared scenario analyses built from package outputs; they are not verified seasonal operating data.

**Unsupported claims.** This notebook does not establish verified Terafab operating data, official Terafab endorsement, actual U.S. national impacts as fact, permitting sufficiency, investment merit, construction, financing, operation, adoption, procurement, verified seasonal forecasts, or empirical validation beyond public-reference screening.